# RK1828 YOLO26-pose Human Pose Estimation Demo

Ultralytics **YOLO26-pose**: detects people and regresses **17 keypoints**
(nose/eyes/ears/shoulders/elbows/wrists/hips/knees/ankles). End-to-end
NMS-free design (we still run a light NMS on the host for dedup), 9 output
heads:

| Head | Shape (one set per stride 8/16/32) | Meaning |
|---|---|---|
| `box_*` | `[1,4,80,80]` `[1,4,40,40]` `[1,4,20,20]` | box regression (l,t,r,b), no DFL |
| `score_*` | `[1,1,...]` | person confidence (single class) |
| `pos_*` | `[1,51,...]` | 17 keypoints x 3 (x, y, confidence logit) |

| Step | Content |
|---|---|
| 1 | Download models (from huggingface.co) |
| 2 | Pick a model and an image |
| 3 | Pre-processing (letterbox: aspect resize + gray pad) |
| 4 | RKNN inference |
| 5 | Post-processing (box + keypoint decode + NMS) |
| 6 | Skeleton visualization |

> Variants: **n/s come in i8 and fp16; m only ships i8** — the m-pose fp16
> weight in the official SDK package is corrupted (a byte-copy of the .rknn
> file), so it is not included.
> Reference: yolo26m-pose i8 ~158 ms/frame, person confidence 0.93 on the sample.

## 0. Environment check

Make sure the RKNN3 runtime (`librknn3_api.so`) and the RK1828 device are ready.
If this fails, install the driver/runtime first (the `rknn3-rk182x` DEB package)
and check with `sudo rknn-smi info` that the RK1828 shows up.

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # loads /usr/lib/librknn3_api.so; failure means the runtime is not installed

devices = find_devices()
if not devices:
    raise RuntimeError("No RK1828 device found! Check: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"Found device: id={dev_id}  interface={dev_type}")

## 1. Download models

The models for this demo are hosted on Hugging Face (account `qhddz`) and are
downloaded directly from **huggingface.co**. Run the next cell to start.

> Files that already exist (and are non-empty) are **skipped automatically** —
> re-running never downloads them twice.
> If your network cannot reach huggingface.co, set up a proxy first, e.g.
> `export https_proxy=http://<your-proxy>:<port>` before starting Jupyter.


This demo downloads YOLO26-pose n/s/m (m is i8-only) and the sample little_boy.jpg.

In [ ]:
# (HF repo, path in repo, local path)
FILES = [
    ("yolo26-pose-rknn3-rk1828", "model/yolo26n-pose_i8.rknn", "models/yolo26_pose/yolo26n-pose_i8.rknn"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26n-pose_i8.weight", "models/yolo26_pose/yolo26n-pose_i8.weight"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26n-pose_fp16.rknn", "models/yolo26_pose/yolo26n-pose_fp16.rknn"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26n-pose_fp16.weight", "models/yolo26_pose/yolo26n-pose_fp16.weight"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26s-pose_i8.rknn", "models/yolo26_pose/yolo26s-pose_i8.rknn"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26s-pose_i8.weight", "models/yolo26_pose/yolo26s-pose_i8.weight"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26s-pose_fp16.rknn", "models/yolo26_pose/yolo26s-pose_fp16.rknn"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26s-pose_fp16.weight", "models/yolo26_pose/yolo26s-pose_fp16.weight"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26m-pose_i8.rknn", "models/yolo26_pose/yolo26m-pose_i8.rknn"),
    ("yolo26-pose-rknn3-rk1828", "model/yolo26m-pose_i8.weight", "models/yolo26_pose/yolo26m-pose_i8.weight"),
    ("yolo26-pose-rknn3-rk1828", "model/little_boy.jpg", "datasets/little_boy.jpg"),
]

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, "hf")

## 2. Pick a model and an image

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_model = widgets.Dropdown(
    options=[("yolo26n-pose i8", "yolo26n-pose_i8"),
             ("yolo26n-pose fp16", "yolo26n-pose_fp16"),
             ("yolo26s-pose i8", "yolo26s-pose_i8"),
             ("yolo26s-pose fp16", "yolo26s-pose_fp16"),
             ("yolo26m-pose i8 (only this one)", "yolo26m-pose_i8")],
    value="yolo26m-pose_i8", description="Model:")
w_image = widgets.Dropdown(options=sorted(glob.glob("datasets/*.jpg")),
                           description="Image:", layout=widgets.Layout(width="60%"))
if any(o.endswith("/little_boy.jpg") for o in w_image.options):
    w_image.value = [o for o in w_image.options if o.endswith("/little_boy.jpg")][0]
display(w_model, w_image)
print("Then keep running downwards ▼")

## 3. Pre-processing: letterbox

YOLO expects a 640x640 square input. Stretching would distort body proportions,
so we **resize proportionally and pad with gray**, remembering the scale and
padding — the post-processing needs them to map boxes and keypoints back.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from vision_post import letterbox, to_input_array, setup_cjk_font

setup_cjk_font()
img = Image.open(w_image.value).convert("RGB")
img_lb, SCALE, PAD_X, PAD_Y = letterbox(img, 640)
x = to_input_array(img_lb)
print(f"original {img.size} -> input {x.shape}, scale={SCALE:.3f}, pad=({PAD_X},{PAD_Y})")

# show every pre-processing stage: original -> letterbox -> input tensor
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, im, title in [
    (axes[0], img, f"1. original {img.size[0]}x{img.size[1]}"),
    (axes[1], img_lb, "2. letterbox 640x640\n(aspect resize + gray pad)"),
    (axes[2], x[0], f"3. model input {x.shape}\nuint8 NHWC"),
]:
    ax.imshow(np.asarray(im))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. RKNN inference

In [ ]:
import time
from rknn3 import Rknn3Model

model = Rknn3Model(f"models/yolo26_pose/{w_model.value}.rknn")
print(f"{len(model.output_attrs)} output heads:")
for a in model.output_attrs:
    print("  ", a)

_ = model.infer([x])
t0 = time.time()
outputs = model.infer([x])
print(f"inference time: {(time.time()-t0)*1000:.1f} ms")

## 5. Post-processing: box + keypoint decode

- Each grid cell: the box head gives (l,t,r,b) distances, the score head the
  person confidence
- Keypoints: `x = (raw_x + grid_x + 0.5) * stride`; confidence goes through sigmoid
- Collect all cells, run one NMS, then map coordinates from letterbox space
  back to the original image

In [ ]:
from vision_post import (decode_yolo26_pose, unletterbox_boxes,
                         unletterbox_keypoints)

boxes, scores, keypoints = decode_yolo26_pose(outputs, model.output_attrs,
                                              conf_thresh=0.4)
boxes = unletterbox_boxes(boxes, SCALE, PAD_X, PAD_Y, *img.size)
keypoints = unletterbox_keypoints(keypoints, SCALE, PAD_X, PAD_Y)

print(f"{len(boxes)} person(s) detected:")
for b, s, k in zip(boxes, scores, keypoints):
    vis = (k[:, 2] >= 0.5).sum()
    print(f"  person {s:.2f}  box=({b[0]:.0f},{b[1]:.0f},{b[2]:.0f},{b[3]:.0f})"
          f"  visible keypoints {vis}/17")

## 6. Skeleton visualization

COCO 17 keypoints connected by the skeleton (shoulder-elbow-wrist,
hip-knee-ankle, etc.); white dots = keypoints with confidence >= 0.5.

In [ ]:
from vision_post import draw_pose

result = draw_pose(img, boxes, scores, keypoints)
model.close()
result

## Summary

- YOLO26-pose outputs keypoints and boxes in the same forward pass (the
  `pos_*` head's 51 channels = 17 points x 3) — no second pose model needed.
- Decode lives in `vision_post.decode_yolo26_pose` / `unletterbox_keypoints`,
  drawing in `vision_post.draw_pose` (`COCO_SKELETON` defines the 17 bones).
- Try n/s for speed, or a multi-person photo to see the skeletons.